# Style Conditioned Language Modeling

Prototype VAE model for style conditioned language modeling.

Goal:
Want a generative model $p_{\theta}(x\mid s)$ which generates text samples $\bar{x}\sim{\theta}(x\mid s)$. We use a VAE to do this. Our data is Shakespeare text. In other words, we will randomly sample the "Shakespeare" style space and produce "Shakespeare" text.

We need:

- Raw data
- Functionality to format the data
- Functionaltiy to tokenize the data

- Create the VAE model using
    - Pytroch Encoder
    - Pytorch Decoder
    - Loss function
    - Training functionality
    - Inference functionality

In [ ]:
## Dataset
## Overwrite Dataset class in Pytorch

from pathlib import Path

from torch.utils.data import Dataset

DATA_DIR = Path("data/shakespeare")

class ShakespeareDataset(Dataset):
    """Raw Shakespeare sentences from the nltk-tokenized ``.original`` files.

    One sentence per line. ``__getitem__`` returns the sentence as a plain
    string; tokenization is handled downstream. ``files`` may be a single
    path or an iterable of paths, whose non-empty lines are concatenated
    in order.
    """

    def __init__(self, files):
        if isinstance(files, (str, Path)):
            files = [files]

        self.lines = []
        for path in files:
            with open(path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line:
                        self.lines.append(line)

    def __len__(self):
        return len(self.lines)

    def __getitem__(self, idx):
        return self.lines[idx]

train = ShakespeareDataset(DATA_DIR / "train.original.nltktok")
validate = ShakespeareDataset(DATA_DIR / "valid.original.nltktok")
test = ShakespeareDataset(DATA_DIR / "test.original.nltktok")


In [ ]:
## Tokenizer
## Word-level vocabulary built from the training split.
## The .nltktok files are already word-tokenized (space separated), so
## "tokenizing" is just split() plus a word -> id lookup.

import torch
from collections import Counter

MAX_VOCAB = None   # None = keep every word in the training split.
                   # Capping at 4000 made <unk> the 2nd most frequent token
                   # in the corpus (6.4%), so the model learned to emit it
                   # constantly. The corpus only has 14,036 unique words.
MAX_LENGTH = 64    # hard cap on sentence length, in tokens

PAD, BOS, EOS, UNK = "<pad>", "<bos>", "<eos>", "<unk>"


class Vocab:
    """Maps words <-> integer ids. Ids 0-3 are reserved for the special tokens."""

    def __init__(self, sentences, max_vocab=MAX_VOCAB):
        # most_common(None) returns every word, ordered by frequency
        counts = Counter(w for s in sentences for w in s.split())
        self.itos = [PAD, BOS, EOS, UNK] + [w for w, _ in counts.most_common(max_vocab)]
        self.stoi = {w: i for i, w in enumerate(self.itos)}
        self.pad_id, self.bos_id, self.eos_id, self.unk_id = 0, 1, 2, 3

    def __len__(self):
        return len(self.itos)

    def encode(self, sentence, max_length=MAX_LENGTH):
        """str -> list[int], wrapped as <bos> ... <eos> and truncated to max_length."""
        ids = [self.stoi.get(w, self.unk_id) for w in sentence.split()]
        return [self.bos_id] + ids[: max_length - 2] + [self.eos_id]

    def decode(self, ids, strip_special=True):
        """list[int] (or a 1-D tensor) -> str. Stops at the first <eos>."""
        words = []
        for i in ids:
            i = int(i)
            if i == self.eos_id:
                break
            if strip_special and i in (self.pad_id, self.bos_id):
                continue
            words.append(self.itos[i])
        return " ".join(words)


vocab = Vocab(train)


def collate_fn(batch):
    """batch: list[str] from ShakespeareDataset -> padded tensors for the model."""
    seqs = [vocab.encode(s) for s in batch]
    T = max(len(s) for s in seqs)
    input_ids = torch.full((len(seqs), T), vocab.pad_id, dtype=torch.long)
    attention_mask = torch.zeros(len(seqs), T, dtype=torch.long)
    for i, s in enumerate(seqs):
        input_ids[i, : len(s)] = torch.tensor(s)
        attention_mask[i, : len(s)] = 1     # 1 = real token, 0 = pad
    return {"input_ids": input_ids, "attention_mask": attention_mask}


print(f"vocab size: {len(vocab)}   example: {vocab.encode(train[0])[:12]}")
print(f"round trip: {vocab.decode(vocab.encode(train[0]))!r}")


In [5]:
## Dataloader

from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_loader = DataLoader(
    train, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, drop_last=True
)
validate_loader = DataLoader(
    validate, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
)
test_loader = DataLoader(
    test, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
)

# peek at one batch
batch = next(iter(train_loader))
print(batch["input_ids"].shape, batch["attention_mask"].shape)
print(batch["input_ids"][0])


torch.Size([32, 57]) torch.Size([32, 57])
tensor([128000,   3923,   4985,    584,    656,   1174,   2998,  32493,  10551,
           949, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001,
        128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001,
        128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001,
        128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001,
        128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001,
        128001, 128001, 128001])


In [ ]:
## Model config
## Hyperparameters + special token ids, used by StyleVAE below

import torch.nn as nn
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

VOCAB_SIZE = len(vocab)
PAD_ID = vocab.pad_id
BOS_ID = vocab.bos_id
EOS_ID = vocab.eos_id
UNK_ID = vocab.unk_id

D_MODEL    = 128   # embedding / hidden width
N_HEAD     = 4     # attention heads
N_LAYERS   = 2     # layers in the encoder, and in the decoder
D_FF       = 256   # feed-forward width
LATENT_DIM = 32    # size of z
DROPOUT    = 0.1
WORD_DROPOUT = 0.25   # decoder-input words replaced by <unk> during training

print(f"vocab={VOCAB_SIZE}  pad={PAD_ID}  bos={BOS_ID}  eos={EOS_ID}  device={DEVICE}")


In [ ]:
## Model Architecture
## Define VAE model with Pytorch transformer encoder and decoder
## Define methods for training and inference

class StyleVAE(nn.Module):
    """Transformer VAE over sentences.

    encoder: tokens -> TransformerEncoder -> masked mean-pool -> (mu, logvar)
    decoder: z -> one "memory" vector -> causal TransformerDecoder -> logits
    """

    def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL, nhead=N_HEAD,
                 num_layers=N_LAYERS, dim_feedforward=D_FF, latent_dim=LATENT_DIM,
                 dropout=DROPOUT, max_len=MAX_LENGTH,
                 pad_id=PAD_ID, bos_id=BOS_ID, eos_id=EOS_ID, unk_id=UNK_ID,
                 word_dropout=WORD_DROPOUT):
        super().__init__()
        self.latent_dim = latent_dim
        self.word_dropout = word_dropout
        self.pad_id, self.bos_id, self.eos_id = pad_id, bos_id, eos_id
        self.unk_id = unk_id

        # token + learned position embeddings, shared by encoder and decoder
        self.embed = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos = nn.Embedding(max_len, d_model)
        self.drop = nn.Dropout(dropout)

        enc_layer = nn.TransformerEncoderLayer(
            d_model, nhead, dim_feedforward, dropout,
            batch_first=True, norm_first=True,
        )
        # norm_first=True is pre-LN, which needs a final LayerNorm after the stack
        self.encoder = nn.TransformerEncoder(
            enc_layer, num_layers, norm=nn.LayerNorm(d_model),
            enable_nested_tensor=False,
        )

        self.to_mu = nn.Linear(d_model, latent_dim)
        self.to_logvar = nn.Linear(d_model, latent_dim)
        self.from_z = nn.Linear(latent_dim, d_model)   # z -> decoder memory

        dec_layer = nn.TransformerDecoderLayer(
            d_model, nhead, dim_feedforward, dropout,
            batch_first=True, norm_first=True,
        )
        self.decoder = nn.TransformerDecoder(
            dec_layer, num_layers, norm=nn.LayerNorm(d_model),
        )

        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.embed.weight        # tie input/output embeddings

        # nn.Embedding defaults to N(0, 1); with tied weights that makes the
        # initial logits huge. Small init => initial loss ~= ln(vocab_size).
        nn.init.normal_(self.embed.weight, std=0.02)
        nn.init.normal_(self.pos.weight, std=0.02)
        with torch.no_grad():
            self.embed.weight[pad_id].zero_()

    def _embed(self, ids):
        pos = torch.arange(ids.size(1), device=ids.device)
        return self.drop(self.embed(ids) + self.pos(pos))

    def encode(self, input_ids, attention_mask):
        """-> mu, logvar, each (B, latent_dim)"""
        h = self.encoder(self._embed(input_ids),
                         src_key_padding_mask=(attention_mask == 0))
        m = attention_mask.unsqueeze(-1).to(h.dtype)          # (B, T, 1)
        pooled = (h * m).sum(1) / m.sum(1).clamp(min=1.0)     # mean over real tokens
        return self.to_mu(pooled), self.to_logvar(pooled)

    def reparameterize(self, mu, logvar):
        """z = mu + sigma * eps -- keeps the sampling differentiable."""
        std = torch.exp(0.5 * logvar)
        return mu + std * torch.randn_like(std)

    def decode(self, z, input_ids, attention_mask=None):
        """Teacher-forced decode. input_ids are decoder *inputs*; -> logits (B, T, V)."""
        memory = self.from_z(z).unsqueeze(1)                  # (B, 1, d_model)
        T = input_ids.size(1)
        causal = torch.ones(T, T, dtype=torch.bool, device=input_ids.device).triu(1)
        h = self.decoder(
            self._embed(input_ids), memory,
            tgt_mask=causal, tgt_is_causal=True,
            tgt_key_padding_mask=None if attention_mask is None else (attention_mask == 0),
        )
        return self.lm_head(h)

    def _word_dropout(self, ids):
        """Randomly replace decoder-input words with <unk> (Bowman et al. 2016).

        The decoder can otherwise predict each word from the previous one and
        ignore z entirely. Blanking words removes that shortcut, so z becomes
        the only reliable signal and the KL rises above the free-bits floor.
        Targets are unaffected -- the model still has to predict the real word.
        """
        if not self.training or self.word_dropout <= 0:
            return ids
        drop = torch.rand(ids.shape, device=ids.device) < self.word_dropout
        drop &= ids != self.bos_id      # always keep <bos>
        drop &= ids != self.pad_id      # leave padding alone
        return ids.masked_fill(drop, self.unk_id)

    def forward(self, input_ids, attention_mask):
        mu, logvar = self.encode(input_ids, attention_mask)
        z = self.reparameterize(mu, logvar)
        # shift: feed x[:, :-1], predict x[:, 1:]
        dec_in = self._word_dropout(input_ids[:, :-1])
        logits = self.decode(z, dec_in, attention_mask[:, :-1])
        targets = input_ids[:, 1:].clone()
        targets[attention_mask[:, 1:] == 0] = -100        # ignore pads in the loss
        return {"logits": logits, "targets": targets, "mu": mu, "logvar": logvar, "z": z}

    @torch.no_grad()
    def generate(self, num_samples=4, max_new_tokens=40, z=None, temperature=1.0,
                 top_k=None):
        """Sample z from the prior N(0, I) and decode autoregressively."""
        self.eval()
        device = next(self.parameters()).device
        if z is None:
            z = torch.randn(num_samples, self.latent_dim, device=device)
        ids = torch.full((z.size(0), 1), self.bos_id, dtype=torch.long, device=device)
        done = torch.zeros(z.size(0), dtype=torch.bool, device=device)
        for _ in range(max_new_tokens):
            logits = self.decode(z, ids)[:, -1] / temperature
            logits[:, self.unk_id] = float("-inf")   # <unk> is a placeholder for
                                                   # missing vocabulary, never a word
            if top_k is not None:                 # keep only the k likeliest words,
                kth = logits.topk(min(top_k, logits.size(-1)), dim=-1).values[:, -1:]
                logits = logits.masked_fill(logits < kth, float("-inf"))
            nxt = torch.multinomial(torch.softmax(logits, dim=-1), 1)
            nxt[done] = self.pad_id
            ids = torch.cat([ids, nxt], dim=1)
            done |= nxt.squeeze(1) == self.eos_id
            if done.all():
                break
        return ids


In [ ]:
## Loss function
## Define the VAE ELBO loss

def elbo_loss(out, beta=1.0, free_bits=0.0):
    """Negative ELBO = reconstruction NLL + beta * KL.

    out       : the dict returned by StyleVAE.forward
    beta      : weight on the KL term. Annealed 0 -> 1 during training. At
                beta=1 this is the true ELBO; starting lower lets the decoder
                learn to use z before the KL pressure can switch it off.
    free_bits : floor (in nats) on each latent dimension's KL. Dimensions
                already below the floor stop being pushed toward the prior,
                which reserves a minimum amount of information for z.

    Both terms are per-sentence sums averaged over the batch, so they are on
    the same scale and can be added directly.
    """
    logits, targets = out["logits"], out["targets"]
    mu, logvar = out["mu"], out["logvar"]
    batch_size = targets.size(0)

    # Reconstruction: -log p(x|z), summed over tokens, averaged over sentences.
    # targets are already -100 at pad positions, so those are skipped.
    recon = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        targets.reshape(-1),
        ignore_index=-100,
        reduction="sum",
    ) / batch_size

    # KL( q(z|x) || N(0, I) ), closed form for diagonal Gaussians.
    kl_dim = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp())   # (B, latent_dim)
    kl_dim = kl_dim.mean(0)                                   # average over batch
    if free_bits > 0:
        kl_dim = kl_dim.clamp(min=free_bits)
    kl = kl_dim.sum()

    n_tokens = (targets != -100).sum().clamp(min=1)
    return {
        "loss": recon + beta * kl,          # backprop this
        "recon": recon.detach(),            # nats per sentence
        "kl": kl.detach(),                  # nats per sentence
        "ppl": (recon * batch_size / n_tokens).exp().detach(),   # per-token perplexity
    }


def beta_schedule(step, warmup_steps, max_beta=1.0):
    """Linear KL annealing: 0 -> max_beta over warmup_steps, then constant.

    Without this the KL term collapses to ~0 in the first few hundred steps,
    the decoder learns to ignore z, and sampling the prior yields the same
    generic sentence every time (posterior collapse).
    """
    return max_beta * min(1.0, step / max(1, warmup_steps))


In [ ]:
## Define the training loop

@torch.no_grad()
def evaluate(model, loader, beta=1.0, free_bits=0.0):
    """Average loss / recon / KL per sentence over a whole loader."""
    model.eval()
    totals = {"loss": 0.0, "recon": 0.0, "kl": 0.0}
    n = 0
    for batch in loader:
        ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        stats = elbo_loss(model(ids, mask), beta=beta, free_bits=free_bits)
        for k in totals:
            totals[k] += float(stats[k]) * ids.size(0)
        n += ids.size(0)
    return {k: v / n for k, v in totals.items()}


@torch.no_grad()
def latent_report(model, loader, threshold=0.01, max_batches=20):
    """Per-dimension KL, and how many latent dimensions are actually used.

    A dimension whose KL is ~0 has q(z_i|x) == the prior: it carries no
    information about x and is dead weight. Counting the "active units"
    is the clearest single diagnostic of what the latent is really doing --
    a model can show a healthy total KL while using only a few dimensions.
    """
    model.eval()
    kls = []
    for i, batch in enumerate(loader):
        if i >= max_batches:
            break
        mu, logvar = model.encode(batch["input_ids"].to(DEVICE),
                                  batch["attention_mask"].to(DEVICE))
        kls.append(-0.5 * (1 + logvar - mu.pow(2) - logvar.exp()))
    kl_dim = torch.cat(kls).mean(0)                      # (latent_dim,)
    return kl_dim, int((kl_dim > threshold).sum())


def train(model, train_loader, valid_loader, epochs=10, lr=3e-4,
          warmup_steps=1000, free_bits=0.2, max_beta=1.0, log_every=500):
    """Train the VAE, printing train stats periodically and validating each epoch.

    free_bits > 0 is what keeps the KL from collapsing to zero. With
    free_bits=0 the decoder learns to ignore z and sampling the prior
    produces the same generic sentence every time.
    """
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    step = 0

    for epoch in range(1, epochs + 1):
        model.train()
        for batch in train_loader:
            ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)

            beta = beta_schedule(step, warmup_steps, max_beta)
            stats = elbo_loss(model(ids, mask), beta=beta, free_bits=free_bits)

            opt.zero_grad()
            stats["loss"].backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            step += 1

            if step % log_every == 0:
                print(f"  step {step:5d}  beta {beta:.2f}"
                      f"  loss {stats['loss'].item():7.2f}"
                      f"  recon {float(stats['recon']):7.2f}"
                      f"  kl {float(stats['kl']):6.2f}"
                      f"  ppl {float(stats['ppl']):7.1f}")

        val = evaluate(model, valid_loader, beta=max_beta, free_bits=free_bits)
        _, active = latent_report(model, valid_loader)
        val["epoch"], val["active_units"] = epoch, active
        history.append(val)
        print(f"epoch {epoch:2d} | valid  loss {val['loss']:7.2f}"
              f"  recon {val['recon']:7.2f}  kl {val['kl']:6.2f}"
              f"  active units {active:2d}/{model.latent_dim}")

    return history


In [ ]:
## Test it all out

torch.manual_seed(0)
model = StyleVAE()
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters on {DEVICE}")
print(f"word dropout: {model.word_dropout}\n")

history = train(model, train_loader, validate_loader, epochs=40, free_bits=0.2)

# Did the posterior collapse? KL near 0 means z is being ignored entirely.
kl = history[-1]["kl"]
print(f"\nfinal validation KL: {kl:.2f} nats -- "
      f"{'COLLAPSED, z is ignored' if kl < 0.5 else 'healthy, z carries information'}")

# Which latent dimensions actually carry information?
kl_dim, active = latent_report(model, validate_loader)
print(f"active units: {active}/{model.latent_dim}   "
      f"(floor is free_bits=0.2 per dim)")
print("per-dim KL, sorted:",
      " ".join(f"{v:.2f}" for v in kl_dim.sort(descending=True).values.tolist()))

# The actual goal: sample the latent space and generate Shakespeare.
print("\n--- samples from the prior N(0, I) ---")
for row in model.generate(num_samples=8, max_new_tokens=20, temperature=0.9, top_k=40):
    print("   ", vocab.decode(row))

# Reconstruction: encode a real sentence, decode from its mu.
batch = next(iter(validate_loader))
ids = batch["input_ids"][:4].to(DEVICE)
mask = batch["attention_mask"][:4].to(DEVICE)
with torch.no_grad():
    mu, _ = model.encode(ids, mask)
recon = model.generate(z=mu, max_new_tokens=20, top_k=40)

print("\n--- reconstructions (encode a sentence, decode from its mu) ---")
for i in range(4):
    print("    in :", vocab.decode(batch["input_ids"][i]))
    print("    out:", vocab.decode(recon[i]))

# Interpolation: walk a straight line between two sentences' latents.
# A smooth, sentence-like path means the latent space actually learned structure.
print("\n--- interpolating between the first two latents ---")
ts = torch.linspace(0, 1, 5, device=DEVICE)
zs = torch.stack([mu[0] + (mu[1] - mu[0]) * t for t in ts])
for t, row in zip(ts, model.generate(z=zs, max_new_tokens=20, top_k=40)):
    print(f"    t={float(t):.2f}  {vocab.decode(row)}")
